# Model Evaluation: Stroke Lesion Segmentation

Full evaluation on ISLES 2022 validation set (50 subjects, fold 0).

**Supports:**
- Standard U-Net and Attention U-Net models (auto-detected from checkpoint)
- Test-time augmentation (TTA) with flip averaging
- Post-processing: small component removal

**Produces:**
- Per-subject metrics (Dice, HD95, volume MAE, lesion F1)
- Stratified analysis by lesion size
- Visualization: Dice distribution, boxplots by size
- Prediction overlays for representative cases

**Setup:**
- Input: `orvile/isles-2022-brain-stoke-dataset`
- Input: Training notebook output (contains best_model.pth)
- GPU T4 recommended (faster inference)

## 1. Setup

In [ ]:
import os
os.environ["MPLBACKEND"] = "Agg"

!pip install -q monai nibabel scipy

%cd /kaggle/working
!rm -rf mri-stroke-assist
!git clone https://gitlab.com/Payz111/mri-stroke-assistance.git mri-stroke-assist
%cd mri-stroke-assist

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Find data & checkpoint

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, "/kaggle/working/mri-stroke-assist")

# --- Find ISLES dataset ---
for candidate in [
    "/kaggle/input/datasets/orvile/isles-2022-brain-stoke-dataset",
    "/kaggle/input/isles-2022-brain-stoke-dataset",
]:
    if os.path.exists(candidate):
        kaggle_input = Path(candidate)
        break
else:
    raise FileNotFoundError("ISLES-2022 dataset not found in /kaggle/input")

isles_root = None
for root in [kaggle_input / "ISLES-2022", kaggle_input]:
    if (root / "sub-strokecase0001").exists():
        isles_root = root
        break
if isles_root is None:
    for root, dirs, files in os.walk(kaggle_input):
        if "sub-strokecase0001" in dirs:
            isles_root = Path(root)
            break

isles_derivatives = isles_root / "derivatives"
print(f"ISLES root: {isles_root}")

# --- Find checkpoint ---
checkpoint_path = None
for item in Path("/kaggle/input").rglob("best_model.pth"):
    checkpoint_path = item
    break

if checkpoint_path is None:
    # Fallback: local repo
    local_ckpt = Path("/kaggle/working/mri-stroke-assist/outputs/fold_0/checkpoints/best_model.pth")
    if local_ckpt.exists():
        checkpoint_path = local_ckpt

if checkpoint_path is None:
    print("Contents of /kaggle/input:")
    for item in Path("/kaggle/input").iterdir():
        print(f"  {item.name}/")
        for sub in list(item.rglob("*.pth"))[:3]:
            print(f"    {sub}")
    raise FileNotFoundError("best_model.pth not found! Add training notebook output as Input.")

print(f"Checkpoint: {checkpoint_path} ({checkpoint_path.stat().st_size / 1e6:.1f} MB)")

## 3. Load model & data

In [ ]:
import json
import yaml
import numpy as np
from src.data.isles22_dataset import ISLES22Dataset
from src.data.transforms import get_val_transforms
from src.models.factory import create_model
from src.eval.metrics import compute_all_metrics
from src.inference.tta import predict_with_tta
from src.eval.postprocess import remove_small_components

# === Configuration ===
USE_TTA = True              # Test-time augmentation (flip averaging)
USE_POSTPROCESS = True      # Remove small false positive clusters
MIN_COMPONENT_SIZE = 10     # Minimum voxels to keep a connected component

print(f"TTA: {USE_TTA}, Post-processing: {USE_POSTPROCESS} (min_size={MIN_COMPONENT_SIZE})")

# Config
cfg_path = Path("/kaggle/working/mri-stroke-assist/configs/default.yaml")
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

device = "cuda" if torch.cuda.is_available() else "cpu"

# Auto-detect model type from checkpoint
# Try attention_unet3d first, fall back to unet3d
model_cfg = dict(cfg["model"])
state = torch.load(checkpoint_path, map_location=device, weights_only=True)

for model_name in ["attention_unet3d", "unet3d"]:
    try:
        model_cfg["name"] = model_name
        model = create_model(model_cfg)
        model.load_state_dict(state)
        print(f"Model loaded: {model_name}")
        break
    except RuntimeError:
        continue
else:
    raise RuntimeError("Could not load checkpoint with any known model architecture")

model.to(device)
model.eval()
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M, device={device}")

# Split
split_file = Path("/kaggle/working/mri-stroke-assist/data/splits/fold_0.json")
with open(split_file) as f:
    split = json.load(f)
print(f"Fold 0: {split['n_train']} train, {split['n_val']} val")

# Val datasets
val_ds_raw = ISLES22Dataset(
    data_root=isles_root,
    derivatives_root=isles_derivatives,
    split_file=split_file,
    split="val",
)
val_ds = ISLES22Dataset(
    data_root=isles_root,
    derivatives_root=isles_derivatives,
    split_file=split_file,
    split="val",
    transform=get_val_transforms(),
)
print(f"Validation set: {len(val_ds)} subjects")

## 4. Run evaluation

In [ ]:
import time
from scipy.ndimage import zoom

results_per_subject = []
predictions = []
ground_truths = []
dwi_volumes = []
subject_ids = []

t0 = time.time()

for i in range(len(val_ds)):
    sample = val_ds[i]
    raw_sample = val_ds_raw[i]
    
    image = sample["image"].unsqueeze(0).to(device)
    # Use TRANSFORMED GT (same 128x128x80 resolution as model output)
    gt_tensor = sample["label"]  # shape: (1, 128, 128, 80)
    
    with torch.no_grad():
        if USE_TTA:
            # TTA: average predictions over original + L-R flip
            pred_prob = predict_with_tta(model, image, flip_axes=[2])
            pred_mask = (pred_prob > 0.5).float()
        else:
            logits = model(image)
            pred_prob = torch.sigmoid(logits)
            pred_mask = (pred_prob > 0.5).float()
    
    pred_np = pred_mask[0, 0].cpu().numpy()   # (128, 128, 80)
    gt_np = (gt_tensor[0].numpy() > 0).astype(np.float32)  # (128, 128, 80)
    
    # Post-processing: remove small connected components
    if USE_POSTPROCESS:
        pred_np = remove_small_components(pred_np, min_size=MIN_COMPONENT_SIZE)
    
    # Use original spacing for physical metrics (HD95, volume in mm/mL)
    spacing = tuple(float(s) for s in raw_sample["metadata"]["spacing"])
    metrics = compute_all_metrics(pred_np, gt_np, spacing)
    
    sid = raw_sample["metadata"]["subject_id"]
    metrics["subject_id"] = sid
    metrics["spacing"] = spacing
    results_per_subject.append(metrics)
    
    predictions.append(pred_np)
    ground_truths.append(gt_np)
    dwi_volumes.append(raw_sample["dwi"])
    subject_ids.append(sid)
    
    if (i + 1) % 10 == 0 or i == 0:
        print(f"[{i+1}/{len(val_ds)}] {sid}: Dice={metrics['dice']:.4f}, "
              f"HD95={metrics['hd95']:.1f}mm, Vol MAE={metrics['volume_mae_ml']:.2f}mL")

elapsed = time.time() - t0
print(f"\nEvaluation complete: {len(val_ds)} subjects in {elapsed:.1f}s ({elapsed/len(val_ds):.1f}s/subject)")
print(f"Settings: TTA={USE_TTA}, PostProcess={USE_POSTPROCESS}")

## 5. Overall metrics

In [ ]:
dices = [r["dice"] for r in results_per_subject]
ious = [r["iou"] for r in results_per_subject]
hd95s = [r["hd95"] for r in results_per_subject if r["hd95"] != float("inf")]
vol_maes = [r["volume_mae_ml"] for r in results_per_subject]
sensitivities = [r["sensitivity"] for r in results_per_subject]
lesion_f1s = [r["lesion_f1"] for r in results_per_subject]

print("=" * 60)
print("OVERALL EVALUATION RESULTS")
print("=" * 60)
print(f"Subjects:    {len(results_per_subject)}")
print(f"Dice:        {np.mean(dices):.4f} +/- {np.std(dices):.4f} (median: {np.median(dices):.4f})")
print(f"IoU:         {np.mean(ious):.4f} +/- {np.std(ious):.4f}")
print(f"HD95:        {np.mean(hd95s):.2f} +/- {np.std(hd95s):.2f} mm (n={len(hd95s)})")
print(f"Vol MAE:     {np.mean(vol_maes):.2f} +/- {np.std(vol_maes):.2f} mL")
print(f"Sensitivity: {np.mean(sensitivities):.4f} +/- {np.std(sensitivities):.4f}")
print(f"Lesion F1:   {np.mean(lesion_f1s):.4f} +/- {np.std(lesion_f1s):.4f}")

## 6. Stratified analysis by lesion size

In [ ]:
# Categorize by GT lesion volume
def size_category(vol_ml):
    if vol_ml < 1:
        return "tiny (<1 mL)"
    elif vol_ml < 10:
        return "small (1-10 mL)"
    elif vol_ml < 50:
        return "medium (10-50 mL)"
    else:
        return "large (>50 mL)"

for r in results_per_subject:
    r["size_cat"] = size_category(r["gt_volume_ml"])

categories = ["tiny (<1 mL)", "small (1-10 mL)", "medium (10-50 mL)", "large (>50 mL)"]

print("\nSTRATIFIED BY LESION SIZE")
print("=" * 70)
print(f"{'Category':<20s} {'N':>4s} {'Dice':>10s} {'HD95 (mm)':>12s} {'Vol MAE':>10s}")
print("-" * 70)

strat_data = {}
for cat in categories:
    cat_results = [r for r in results_per_subject if r["size_cat"] == cat]
    n = len(cat_results)
    if n == 0:
        print(f"{cat:<20s} {0:>4d}   {'--':>8s}   {'--':>10s}   {'--':>8s}")
        continue
    d = [r["dice"] for r in cat_results]
    h = [r["hd95"] for r in cat_results if r["hd95"] != float("inf")]
    v = [r["volume_mae_ml"] for r in cat_results]
    print(f"{cat:<20s} {n:>4d}   {np.mean(d):.4f}+/-{np.std(d):.3f}   "
          f"{np.mean(h) if h else float('nan'):>6.1f}+/-{np.std(h) if h else 0:.1f}   "
          f"{np.mean(v):.2f}+/-{np.std(v):.2f}")
    strat_data[cat] = {"dices": d, "hd95s": h, "vol_maes": v}

## 7. Visualization: metrics distribution

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Dice histogram
axes[0].hist(dices, bins=20, color="steelblue", edgecolor="white", alpha=0.8)
axes[0].axvline(np.mean(dices), color="red", linestyle="--", label=f"Mean: {np.mean(dices):.3f}")
axes[0].axvline(np.median(dices), color="orange", linestyle="--", label=f"Median: {np.median(dices):.3f}")
axes[0].set_xlabel("Dice Score")
axes[0].set_ylabel("Count")
axes[0].set_title("Dice Score Distribution")
axes[0].legend()

# 2. Dice by lesion size (boxplot)
box_data = []
box_labels = []
for cat in categories:
    if cat in strat_data:
        box_data.append(strat_data[cat]["dices"])
        # Short labels for boxplot
        short = cat.split(" (")[0]
        n = len(strat_data[cat]["dices"])
        box_labels.append(f"{short}\n(n={n})")

bp = axes[1].boxplot(box_data, labels=box_labels, patch_artist=True)
colors = ["#ff9999", "#66b3ff", "#99ff99", "#ffcc99"]
for patch, color in zip(bp["boxes"], colors[:len(box_data)]):
    patch.set_facecolor(color)
axes[1].set_ylabel("Dice Score")
axes[1].set_title("Dice by Lesion Size")
axes[1].grid(axis="y", alpha=0.3)

# 3. Predicted vs GT volume scatter
pred_vols = [r["pred_volume_ml"] for r in results_per_subject]
gt_vols = [r["gt_volume_ml"] for r in results_per_subject]
max_vol = max(max(pred_vols), max(gt_vols)) * 1.1
axes[2].scatter(gt_vols, pred_vols, alpha=0.6, c="steelblue", edgecolors="white", s=50)
axes[2].plot([0, max_vol], [0, max_vol], "r--", alpha=0.5, label="Perfect")
axes[2].set_xlabel("Ground Truth Volume (mL)")
axes[2].set_ylabel("Predicted Volume (mL)")
axes[2].set_title("Volume: Predicted vs Ground Truth")
axes[2].legend()
axes[2].set_xlim(0, max_vol)
axes[2].set_ylim(0, max_vol)
axes[2].set_aspect("equal")

plt.tight_layout()
plt.savefig("/kaggle/working/eval_metrics.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: /kaggle/working/eval_metrics.png")

## 8. Prediction overlays (representative cases)

In [ ]:
# Pick representative cases: best, worst, and one per size category
sorted_by_dice = sorted(range(len(results_per_subject)), key=lambda i: results_per_subject[i]["dice"])

# Indices to show
show_indices = set()
show_indices.add(sorted_by_dice[-1])  # best
show_indices.add(sorted_by_dice[-2])  # 2nd best
show_indices.add(sorted_by_dice[len(sorted_by_dice)//2])  # median
show_indices.add(sorted_by_dice[0])   # worst

# One per size category
for cat in categories:
    cat_indices = [i for i, r in enumerate(results_per_subject) if r["size_cat"] == cat]
    if cat_indices:
        cat_dices = [results_per_subject[i]["dice"] for i in cat_indices]
        median_dice = np.median(cat_dices)
        closest = min(cat_indices, key=lambda i: abs(results_per_subject[i]["dice"] - median_dice))
        show_indices.add(closest)

show_indices = sorted(show_indices)
print(f"Showing {len(show_indices)} representative cases")

for idx in show_indices:
    r = results_per_subject[idx]
    pred = predictions[idx]   # 128x128x80
    gt = ground_truths[idx]   # 128x128x80
    dwi = dwi_volumes[idx]    # original resolution
    
    # Handle 4D
    if dwi.ndim == 4:
        dwi = dwi[..., 0]
    
    # Resize pred & GT to DWI shape for visualization only
    dwi_shape = dwi.shape
    if pred.shape != dwi_shape:
        factors = [d / p for d, p in zip(dwi_shape, pred.shape)]
        pred_vis = zoom(pred, factors, order=0)[:dwi_shape[0], :dwi_shape[1], :dwi_shape[2]]
        gt_vis = zoom(gt, factors, order=0)[:dwi_shape[0], :dwi_shape[1], :dwi_shape[2]]
    else:
        pred_vis = pred
        gt_vis = gt
    
    # Find best slice (most GT lesion voxels)
    gt_per_slice = gt_vis.sum(axis=(0, 1))
    if gt_per_slice.max() > 0:
        best_z = int(np.argmax(gt_per_slice))
    else:
        pred_per_slice = pred_vis.sum(axis=(0, 1))
        best_z = int(np.argmax(pred_per_slice)) if pred_per_slice.max() > 0 else dwi.shape[2] // 2
    
    # Show 3 slices
    n_slices = 3
    slices = [max(0, min(dwi.shape[2]-1, best_z + i - 1)) for i in range(n_slices)]
    
    fig, axes = plt.subplots(2, n_slices, figsize=(12, 8))
    fig.suptitle(
        f"{r['subject_id']} | Dice={r['dice']:.3f} | "
        f"GT={r['gt_volume_ml']:.1f}mL | Pred={r['pred_volume_ml']:.1f}mL | "
        f"{r['size_cat']}",
        fontsize=12, fontweight="bold"
    )
    
    # Normalize DWI
    p1, p99 = np.percentile(dwi[dwi > 0], [1, 99]) if dwi.any() else (0, 1)
    dwi_norm = np.clip((dwi - p1) / (p99 - p1 + 1e-8), 0, 1)
    
    for i, z in enumerate(slices):
        # Row 1: DWI + Ground Truth (green)
        axes[0, i].imshow(dwi_norm[:, :, z].T, cmap="gray", origin="lower")
        if gt_vis[:, :, z].sum() > 0:
            axes[0, i].imshow(gt_vis[:, :, z].T, cmap="Greens", alpha=0.4, origin="lower")
        axes[0, i].set_title(f"GT z={z}", fontsize=10)
        axes[0, i].axis("off")
        
        # Row 2: DWI + Prediction (red)
        axes[1, i].imshow(dwi_norm[:, :, z].T, cmap="gray", origin="lower")
        if pred_vis[:, :, z].sum() > 0:
            axes[1, i].imshow(pred_vis[:, :, z].T, cmap="Reds", alpha=0.4, origin="lower")
        axes[1, i].set_title(f"Pred z={z}", fontsize=10)
        axes[1, i].axis("off")
    
    axes[0, 0].set_ylabel("Ground Truth", fontsize=11, fontweight="bold")
    axes[1, 0].set_ylabel("Prediction", fontsize=11, fontweight="bold")
    
    plt.tight_layout()
    fname = f"/kaggle/working/overlay_{r['subject_id']}.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {fname}")

## 9. Save results

In [ ]:
# Save per-subject results as JSON
output = {
    "model": "v1.0-unet3d-isles22-soop",
    "checkpoint": str(checkpoint_path),
    "n_subjects": len(results_per_subject),
    "overall": {
        "dice_mean": float(np.mean(dices)),
        "dice_std": float(np.std(dices)),
        "dice_median": float(np.median(dices)),
        "iou_mean": float(np.mean(ious)),
        "hd95_mean": float(np.mean(hd95s)) if hd95s else None,
        "hd95_std": float(np.std(hd95s)) if hd95s else None,
        "volume_mae_mean": float(np.mean(vol_maes)),
        "sensitivity_mean": float(np.mean(sensitivities)),
        "lesion_f1_mean": float(np.mean(lesion_f1s)),
    },
    "per_subject": [
        {k: (float(v) if isinstance(v, (np.floating, float)) else v)
         for k, v in r.items()}
        for r in results_per_subject
    ],
}

with open("/kaggle/working/eval_results.json", "w") as f:
    json.dump(output, f, indent=2)

print("Saved: /kaggle/working/eval_results.json")
print(f"\nOutput files:")
for f in Path("/kaggle/working").glob("eval_*"):
    print(f"  {f.name} ({f.stat().st_size / 1e3:.1f} KB)")
for f in Path("/kaggle/working").glob("overlay_*"):
    print(f"  {f.name} ({f.stat().st_size / 1e3:.1f} KB)")

## Summary

Results saved to `/kaggle/working/`:
- `eval_results.json` -- per-subject metrics
- `eval_metrics.png` -- Dice distribution + boxplots + volume scatter
- `overlay_*.png` -- Prediction vs GT overlays for representative cases

Download these for README and docs.